<a href="https://colab.research.google.com/github/MayerT1/LiDAR_Dev/blob/main/Covarite_Export_Sewanee_Pixel_FUSION_GEDI_RS_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.mdpi.com/2072-4292/13/3/442
https://iopscience.iop.org/article/10.1088/1748-9326/ab9e99/meta
https://dataverse.jpl.nasa.gov/file.xhtml?fileId=64312&version=3.0

https://www.neonscience.org/field-sites/guan
https://www.neonscience.org/field-sites/guan#field-site-data
https://data.neonscience.org/data-products/explore
https://data.neonscience.org/data-products/DP1.10098.001

https://lpdaac.usgs.gov/documents/980/gedi_l2b_dictionary_P003_v2.html

# Setup

In [ ]:
!pip install gdal
!pip install laspy
!pip install lazrs
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.6/639.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 97.2 MB/s eta 0:00:00


In [ ]:
# Install necessary packages
!pip install geemap geopandas scikit-learn tensorflow

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
import tensorflow as tf
from tensorflow.keras import layers, models


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.7 MB/s eta 0:00:00


In [ ]:
# Authenticate and initialize Earth Engine
import ee
import geemap
import geemap.chart as chart
ee.Authenticate()
ee.Initialize(project='servir-sco-assets')
Map = geemap.Map()


In [ ]:

from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import os

# Define the base project folder and subdirectories
base_dir = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project'
subdirs = [
    'data',
    'models',
    'scripts',
    'notebooks',
    'config',
    'results'
]

# Create each subdirectory
for subdir in subdirs:
    path = os.path.join(base_dir, subdir)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")


Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/models
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/scripts
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/notebooks
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/config
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/results


In [ ]:
%cd /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project

/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project


In [ ]:

s1Ascending_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021")
DEMindices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021")
HLS_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/HLS_2021")
LandsatComposite_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/LandsatComposite_2021")
LandsatIndices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/LandsatIndices_2021")
S2Composite_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Composite_2021")
S2Indices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Indices_2021")
landsatTasseledCapIndices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/landsatTasseledCapIndices_2021")
ROI = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_Domain')
# GEDIindicesA_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/GEDIindicesA_2021")
# GEDIindicesB_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/GEDIindicesB_2021")

Map.addLayer(ROI, {}, 'ROI')
Map.addLayer(s1Ascending_2021, {}, 's1Ascending_2021')
Map.addLayer(DEMindices_2021, {}, 'DEMindices_2021')
Map.addLayer(GEDIindicesA_2021, {}, 'GEDIindicesA_2021')
Map.addLayer(GEDIindicesB_2021, {}, 'GEDIindicesB_2021')
Map.addLayer(HLS_2021, {}, 'HLS_2021')
Map.addLayer(LandsatComposite_2021, {}, 'LandsatComposite_2021')
Map.addLayer(LandsatIndices_2021, {}, 'LandsatIndices_2021')
Map.addLayer(S2Composite_2021, {}, 'S2Composite_2021')
Map.addLayer(S2Indices_2021, {}, 'S2Indices_2021')
Map.addLayer(landsatTasseledCapIndices_2021, {}, 'landsatTasseledCapIndices_2021')

stacked = s1Ascending_2021.addBands(DEMindices_2021).addBands(LandsatComposite_2021).addBands(LandsatIndices_2021).addBands(landsatTasseledCapIndices_2021).addBands(HLS_2021).addBands(S2Composite_2021).addBands(S2Indices_2021)

Map.addLayer(stacked, {}, 'stacked');

Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

#Spatial Autocorreltion Block

In [ ]:
#//////////////////////////////////////////////////////////////////////////////////
GEDIindicesA_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/GEDIindicesA_2021")


glad = ee.Image("projects/glad/GLCLU2020/Forest_height_2020").clip(ROI)
Map.addLayer(glad, {}, "glad")
glad = glad.gte(1).select(['b1']).rename(['class']);
#print("glad", glad)

classMask = GEDIindicesA_2021.addBands(glad)
Map.addLayer(classMask, {}, "classMask")
#print("classMask", classMask)


proj = GEDIindicesA_2021.projection()
grid = ROI.geometry().coveringGrid(proj, 30)
grid = ee.FeatureCollection(grid).randomColumn("random", 42);
#print("spatial_partition number of boxes in the grid", grid.size())
val_samp = grid.filter('random <= 0.1').set("samp_type","val_samp");
test_samp = grid.filter('random <= 0.3 and random >= 0.1').set("samp_type","test_samp");
train_samp = grid.filter('random >= 0.3').set("samp_type","train_samp");


#rint('val_samp 10%', val_samp.size());
Map.addLayer(val_samp, {"color": "blue"}, "val_samp 10%")

#print('test_samp 20%', test_samp.size());
Map.addLayer(test_samp, {"color": "red"}, "test_samp 20%")

#print('train_samp 70%', train_samp.size());
Map.addLayer(train_samp, {"color": "green"}, "train_samp 70%")

Map

Map(bottom=207702.0, center=[35.15023157128286, -85.76407231260058], controls=(WidgetControl(options=['positio…

In [ ]:
import ee
import geemap
import numpy as np
from tqdm import tqdm
import os
import json

# ee.Initialize()

# Load individual EO inputs (2021)
s1Ascending_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021")
DEMindices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/DEMindices_2021")
HLS_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/HLS_2021")
LandsatComposite_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/LandsatComposite_2021")
LandsatIndices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/LandsatIndices_2021")
landsatTasseledCapIndices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/landsatTasseledCapIndices_2021")
S2Composite_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Composite_2021")
S2Indices_2021 = ee.Image("projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Indices_2021")

# ROI
ROI = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_Domain')

# Sampling Grid (based on one input image’s projection)
scale = 20
proj = LandsatComposite_2021.select(0).projection()
grid = ROI.geometry().coveringGrid(proj, scale)
grid = ee.FeatureCollection(grid).randomColumn("random", 42)

# Partition
val_samp = grid.filter('random <= 0.1')
test_samp = grid.filter('random > 0.1 and random <= 0.3')
train_samp = grid.filter('random > 0.3')

# Optional: Downsample geometries for performance
def sample_fc_randomly(fc, percent=0.1, seed=42):
    fc = fc.randomColumn("rand", seed)
    return fc.filter(ee.Filter.lt("rand", percent))

SAMPLE_PERCENT = 0.0001
train_samp = sample_fc_randomly(train_samp, SAMPLE_PERCENT)
val_samp = sample_fc_randomly(val_samp, SAMPLE_PERCENT)
test_samp = sample_fc_randomly(test_samp, SAMPLE_PERCENT)

# Output directory on Google Drive
data_dir = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data'
os.makedirs(data_dir, exist_ok=True)

# Extract numpy arrays and geometry metadata
def extract_numpy_with_geometry(image, fc, scale, name="set"):
    bands = image.bandNames().getInfo()
    features = fc.toList(fc.size())
    arrays = []
    metadata = []

    for i in tqdm(range(fc.size().getInfo()), desc=f"{name} - {image.getInfo().get('id', 'image')}"):
        try:
            feat = ee.Feature(features.get(i))
            geom = feat.geometry()
            arr = geemap.ee_to_numpy(image, region=geom, scale=scale, bands=bands)

            if arr is not None and arr.shape[0] > 0 and arr.shape[1] > 0:
                arrays.append(arr)
                metadata.append(geom.getInfo())
        except Exception as e:
            print(f"Skipped {name} tile {i}: {e}")
            continue

    return np.array(arrays), metadata

# Save numpy and metadata
def save_numpy_and_metadata(arrays, metadata, prefix):
    npy_path = os.path.join(data_dir, f"{prefix}.npy")
    meta_path = os.path.join(data_dir, f"{prefix}_geom.json")

    np.save(npy_path, arrays)
    with open(meta_path, 'w') as f:
        json.dump(metadata, f)

    print(f"Saved: {npy_path} and {meta_path}")

# List of input sources
input_images = {
    "s1": s1Ascending_2021,
    "dem": DEMindices_2021,
    "hls": HLS_2021,
    "landsat": LandsatComposite_2021,
    "landsat_idx": LandsatIndices_2021,
    "tcap": landsatTasseledCapIndices_2021,
    "s2": S2Composite_2021,
    "s2_idx": S2Indices_2021,
}

# Extract and save everything
for key, img in input_images.items():
    for split_name, fc in zip(["train", "val", "test"], [train_samp, val_samp, test_samp]):
        arrays, metadata = extract_numpy_with_geometry(img, fc, scale, name=f"{key}_{split_name}")
        save_numpy_and_metadata(arrays, metadata, prefix=f"{key}_{split_name}")


s1_train - projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021: 100%|██████████| 11/11 [08:57<00:00, 48.83s/it]


Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s1_train.npy and /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s1_train_geom.json


s1_val - projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021: 0it [00:00, ?it/s]


Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s1_val.npy and /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s1_val_geom.json


s1_test - projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/s1Ascending_2021: 100%|██████████| 7/7 [04:35<00:00, 39.38s/it]


...

Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_train.npy and /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_train_geom.json


s2_idx_val - projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Indices_2021: 0it [00:00, ?it/s]


Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_val.npy and /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_val_geom.json


s2_idx_test - projects/servir-sco-assets/assets/Rx_Fire/EO_Outputs/Sewanee/S2Indices_2021: 100%|██████████| 7/7 [04:16<00:00, 36.65s/it]

Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_test.npy and /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data/s2_idx_test_geom.json
